In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader

In [2]:
FAMILY_NAME = "modulo_addition_1layer"
EXPORT_DIR = Path("exports/weight_pca_scatter")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# (prime, model_seed, data_seed, label)
MODELS = [
    (109, 485, 598, "p109 reference healthy"),
    (59,  485, 598, "p59 incomplete freq set"),
    (113, 999, 598, "p113 canon (diffuse)"),
    (101, 999, 598, "p101 open loop"),
    (59, 485, 999, "p59/s485/ds999 not-yet-grokked "),
    (59, 999, 999, "p59/s999/ds999 degraded "),
    (89, 999, 999, "p89/s999/ds999 degraded "),
]

family = load_family(FAMILY_NAME)

## Whole-matrix W_in PCA — per-epoch snapshot

Fresh PCA on the full W_in at a single epoch. Each neuron is a point in the weight space; the PCA basis is refit at the chosen epoch so the axes reflect the intrinsic geometry at that moment.

Neurons are colored by their frequency group assignment (from the final-epoch neuron_group_pca artifact).

In [3]:
import colorsys

def _freq_color(freq_idx: int, n_freq: int) -> str:
    hue = freq_idx / max(n_freq, 1)
    r, g, b = colorsys.hls_to_rgb(hue, 0.55, 0.5)
    return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"


def load_whole_matrix_data(variant, epoch=None):
    """Load W_in at `epoch` plus neuron group labels from neuron_group_pca.

    Returns dict:
        W_in              (d_model, d_mlp) — weight matrix at requested epoch
        epoch             int — actual epoch used
        neuron_group_idx  (d_mlp,) int — group index per neuron (-1 = ungrouped)
        group_freqs       (n_groups,) int — frequency index per group
        prime             int
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))

    # resolve epoch
    available = sorted(loader.get_epochs("parameter_snapshot"))
    if epoch is None:
        epoch = available[-1]
    else:
        epoch = available[min(range(len(available)), key=lambda i: abs(available[i] - epoch))]

    snap = loader.load_epoch("parameter_snapshot", epoch)
    group_data = loader.load_cross_epoch("neuron_group_pca")

    return {
        "W_in": snap["W_in"],                              # (d_model, d_mlp)
        "epoch": epoch,
        "neuron_group_idx": group_data["neuron_group_idx"],
        "group_freqs": group_data["group_freqs"],
        "prime": int(variant.model_config["prime"]),
    }


def whole_matrix_pca(W_in: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """PCA on the full W_in, treating each neuron as a point in weight space.

    Args:
        W_in: (d_model, d_mlp) — neurons are columns

    Returns:
        coords:       (d_mlp, 3) — PC1/PC2/PC3 coordinates for each neuron
        var_explained (3,)       — fraction of variance per component
    """
    X = W_in.T                          # (d_mlp, d_model) — neurons as rows
    X = X - X.mean(axis=0)              # center
    _, S, Vt = np.linalg.svd(X, full_matrices=False)
    coords = X @ Vt[:3].T               # (d_mlp, 3)
    var_explained = (S**2 / (S**2).sum())[:3]
    return coords, var_explained

In [4]:
def plot_win_global_pca(variant, epoch=None, title_prefix="") -> go.Figure:
    """3D scatter of all W_in neurons in global PCA space at a single epoch.

    Unlike neuron_group.scatter_3d (per-group PCA, centroid-centered),
    this uses a single PCA basis fit to the whole W_in. Neurons are colored
    by their final-epoch frequency group assignment.
    """
    data = load_whole_matrix_data(variant, epoch)
    coords, var_exp = whole_matrix_pca(data["W_in"])

    neuron_group_idx = data["neuron_group_idx"]
    group_freqs = data["group_freqs"]
    prime = data["prime"]
    actual_epoch = data["epoch"]
    n_freq = int(group_freqs.max()) + 1 if len(group_freqs) > 0 else 1

    fig = go.Figure()

    # one trace per frequency group
    for g_idx, freq in enumerate(group_freqs):
        members = np.where(neuron_group_idx == g_idx)[0]
        if len(members) == 0:
            continue
        pts = coords[members]
        color = _freq_color(int(freq), n_freq)
        fig.add_trace(go.Scatter3d(
            x=pts[:, 0].tolist(),
            y=pts[:, 1].tolist(),
            z=pts[:, 2].tolist(),
            mode="markers",
            name=f"freq {int(freq) + 1} (n={len(members)})",
            marker=dict(size=4, color=color, opacity=0.8),
            hovertemplate=(
                f"freq {int(freq) + 1}<br>"
                "neuron %{customdata}<br>"
                "PC1=%{x:.3f} PC2=%{y:.3f} PC3=%{z:.3f}<extra></extra>"
            ),
            customdata=members.tolist(),
        ))

    # ungrouped neurons (neuron_group_idx == -1)
    ungrouped = np.where(neuron_group_idx == -1)[0]
    if len(ungrouped) > 0:
        pts = coords[ungrouped]
        fig.add_trace(go.Scatter3d(
            x=pts[:, 0].tolist(),
            y=pts[:, 1].tolist(),
            z=pts[:, 2].tolist(),
            mode="markers",
            name=f"ungrouped (n={len(ungrouped)})",
            marker=dict(size=3, color="lightgray", opacity=0.4),
        ))

    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]
    title = f"{title_prefix}p={prime} | epoch {actual_epoch} | W_in global PCA"

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title=axis_labels[0],
            yaxis_title=axis_labels[1],
            zaxis_title=axis_labels[2],
        ),
        legend=dict(x=1.01, y=1, font=dict(size=10)),
        template="plotly_white",
        height=650,
    )
    return fig

In [5]:
# --- Run: p109 reference healthy, final epoch ---
prime, model_seed, data_seed, label = MODELS[6]
v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)

fig = plot_win_global_pca(v, epoch=None, title_prefix=f"{label} | ")
fig.show()

## Animated W_in global PCA — epoch scrubber

PCA basis fixed to the final epoch so axes are stable across frames.
Each epoch's W_in is centered and projected through that shared basis.

In [6]:
def load_epoch_projections(variant, n_frames=60, basis_group_index: int=0):
    """Load W_in at evenly-spaced epochs and project through final-epoch PCA basis.

    Using a fixed basis (fit at final epoch) keeps axes stable across frames
    so the animation shows structural change, not axis drift.

    Returns dict:
        epochs        list[int]             — epochs used (length <= n_frames)
        coords        dict[int, np.ndarray] — epoch → (d_mlp, 3) projected coords
        basis         (3, d_model)          — PCA basis vectors (final epoch)
        center        (d_model,)            — centering vector (final epoch global mean)
        var_explained (3,)                  — variance explained by each PC (final epoch)
        neuron_group_idx  (d_mlp,)
        group_freqs       (n_groups,)
        prime         int
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    available = sorted(loader.get_epochs("parameter_snapshot"))

    # subsample evenly, always include final epoch
    indices = np.linspace(0, len(available) - 1, n_frames, dtype=int)
    epochs = sorted(set(available[i] for i in indices))

    group_data = loader.load_cross_epoch("neuron_group_pca")

    # fit basis on final epoch
    snap_final = loader.load_epoch("parameter_snapshot", available[-1])
    W_final = snap_final["W_in"]          # (d_model, d_mlp)
    X_final = W_final.T                   # (d_mlp, d_model)
    center = X_final.mean(axis=0)
    X_centered = X_final - center
    _, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    basis_start_index = 1 * basis_group_index
    basis_end_index = basis_start_index + 3 
    
    basis = Vt[basis_start_index:basis_end_index:1] # (3, d_model)
    var_explained = (S**2 / (S**2).sum())[basis_start_index:basis_end_index:1]

    # project each epoch through the fixed basis
    coords = {}
    for epoch in epochs:
        snap = loader.load_epoch("parameter_snapshot", epoch)
        X = snap["W_in"].T - center       # same centering
        coords[epoch] = X @ basis.T       # (d_mlp, 3)

    return {
        "epochs": epochs,
        "coords": coords,
        "basis": basis,
        "center": center,
        "var_explained": var_explained,
        "neuron_group_idx": group_data["neuron_group_idx"],
        "group_freqs": group_data["group_freqs"],
        "prime": int(variant.model_config["prime"]),
    }

In [7]:
def _group_traces(coords_ep, neuron_group_idx, group_freqs, n_freq):
    """Build one Scatter3d trace per group for a single epoch's coords."""
    traces = []

    for g_idx, freq in enumerate(group_freqs):
        members = np.where(neuron_group_idx == g_idx)[0]
        if len(members) == 0:
            continue
        pts = coords_ep[members]
        color = _freq_color(int(freq), n_freq)
        traces.append(go.Scatter3d(
            x=pts[:, 0].tolist(),
            y=pts[:, 1].tolist(),
            z=pts[:, 2].tolist(),
            mode="markers",
            name=f"freq {int(freq) + 1} (n={len(members)})",
            marker=dict(size=4, color=color, opacity=0.8),
            showlegend=True,
        ))
    ungrouped = np.where(neuron_group_idx == -1)[0]
    if len(ungrouped) > 0:
        pts = coords_ep[ungrouped]
        traces.append(go.Scatter3d(
            x=pts[:, 0].tolist(),
            y=pts[:, 1].tolist(),
            z=pts[:, 2].tolist(),
            mode="markers",
            name=f"ungrouped (n={len(ungrouped)})",
            marker=dict(size=3, color="lightgray", opacity=0.3),
            showlegend=True,
        ))
    return traces


def build_win_animation(proj: dict, title_prefix="") -> go.Figure:
    """Animated 3D scatter of W_in neurons with an epoch scrubber.

    Args:
        proj: output of load_epoch_projections
    """
    epochs = proj["epochs"]
    coords = proj["coords"]
    neuron_group_idx = proj["neuron_group_idx"]
    group_freqs = proj["group_freqs"]
    var_exp = proj["var_explained"]
    prime = proj["prime"]
    n_freq = int(group_freqs.max()) + 1 if len(group_freqs) > 0 else 1

    # compute axis ranges from all epochs so camera doesn't jump
    all_pts = np.vstack(list(coords.values()))
    pad = 0.05
    def _range(col):
        lo, hi = all_pts[:, col].min(), all_pts[:, col].max()
        margin = (hi - lo) * pad
        return [lo - margin, hi + margin]

    axis_labels = [f"PC{i+1} ({v:.1%})" for i, v in enumerate(var_exp)]

    # initial frame (first epoch)
    initial_traces = _group_traces(coords[epochs[0]], neuron_group_idx, group_freqs, n_freq)

    # build frames
    frames = [
        go.Frame(
            data=_group_traces(coords[ep], neuron_group_idx, group_freqs, n_freq),
            name=str(ep),
        )
        for ep in epochs
    ]

    slider_steps = [
        {
            "args": [[str(ep)], {"frame": {"duration": 0}, "mode": "immediate",
                                 "transition": {"duration": 0}}],
            "label": str(ep),
            "method": "animate",
        }
        for ep in epochs
    ]

    fig = go.Figure(
        data=initial_traces,
        frames=frames,
        layout=go.Layout(
            title=f"{title_prefix}p={prime} | W_in global PCA (fixed final-epoch basis)",
            scene=dict(
                xaxis=dict(title=axis_labels[0], range=_range(0)),
                yaxis=dict(title=axis_labels[1], range=_range(1)),
                zaxis=dict(title=axis_labels[2], range=_range(2)),
            ),
            legend=dict(x=1.01, y=1, font=dict(size=10)),
            template="plotly_white",
            height=700,
            updatemenus=[{
                "buttons": [
                    {"args": [None, {"frame": {"duration": 150}, "fromcurrent": True,
                                     "transition": {"duration": 0}}],
                     "label": "Play", "method": "animate"},
                    {"args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}],
                     "label": "Pause", "method": "animate"},
                ],
                "type": "buttons",
                "showactive": False,
                "x": 0.05, "y": 0, "xanchor": "left", "yanchor": "top",
            }],
            sliders=[{
                "steps": slider_steps,
                "currentvalue": {"prefix": "Epoch: ", "font": {"size": 13}},
                "pad": {"t": 50},
                "len": 0.85,
                "x": 0.1,
            }],
        ),
    )
    return fig

In [ ]:
# Can only show 3 components at time in the 3D Scatter.
# Basis group index allows choosing sets of 3 at a time 
# to see if more dimensions are being used
basis_group_index = 0

# --- Run: p109 reference healthy, 60 frames ---
prime, model_seed, data_seed, label = MODELS[1]
v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)

proj = load_epoch_projections(v, n_frames=60, basis_group_index=basis_group_index)
print(f"Loaded {len(proj['epochs'])} epochs: {proj['epochs'][0]} → {proj['epochs'][-1]}")

fig = build_win_animation(proj, title_prefix=f"{label} | ")
fig.show()

Loaded 60 epochs: 0 → 34999


TypeError: build_win_animation() got an unexpected keyword argument 'basis_group_index'

## Animated in-group geometry — epoch scrubber

Per-group PCA view animated over training. Coordinates come directly from the
pre-computed `projections` field in the `neuron_group_pca` artifact — each neuron
is already expressed in its group's centroid-centered, final-epoch PCA basis.
No W_in reload needed per frame.

In [ ]:
def build_ingroup_animation(variant, n_frames=60, title_prefix="") -> go.Figure:
    """Animated in-group PCA scatter with an epoch scrubber.

    Uses the pre-computed projections from the neuron_group_pca artifact.
    Each neuron's coordinates are in its own group's centroid-centered,
    final-epoch PCA space — the same coordinate system as neuron_group.scatter_3d.
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    data = loader.load_cross_epoch("neuron_group_pca")

    all_epochs = data["epochs"].tolist()          # (n_epochs,)
    projections = data["projections"]              # (n_epochs, d_mlp, 3)
    neuron_group_idx = data["neuron_group_idx"]   # (d_mlp,)
    group_freqs = data["group_freqs"]             # (n_groups,)
    prime = int(variant.model_config["prime"])
    n_freq = int(group_freqs.max()) + 1 if len(group_freqs) > 0 else 1

    # subsample epoch indices evenly
    idx_sample = np.linspace(0, len(all_epochs) - 1, n_frames, dtype=int)
    sampled = sorted(set(idx_sample.tolist()))

    # axis ranges fixed across all sampled frames
    all_pts = projections[sampled].reshape(-1, 3)
    valid = all_pts[~np.isnan(all_pts).any(axis=1)]
    pad = 0.05
    def _range(col):
        lo, hi = valid[:, col].min(), valid[:, col].max()
        margin = (hi - lo) * pad
        return [lo - margin, hi + margin]

    def _traces_at(ep_idx):
        pts = projections[ep_idx]   # (d_mlp, 3)
        traces = []
        for g_idx, freq in enumerate(group_freqs):
            members = np.where(neuron_group_idx == g_idx)[0]
            if len(members) == 0:
                continue
            g_pts = pts[members]
            valid_mask = ~np.isnan(g_pts).any(axis=1)
            g_pts = g_pts[valid_mask]
            color = _freq_color(int(freq), n_freq)
            traces.append(go.Scatter3d(
                x=g_pts[:, 0].tolist(),
                y=g_pts[:, 1].tolist(),
                z=g_pts[:, 2].tolist(),
                mode="markers",
                name=f"freq {int(freq) + 1} (n={len(members)})",
                marker=dict(size=4, color=color, opacity=0.8),
                showlegend=True,
            ))
        return traces

    frames = [
        go.Frame(data=_traces_at(i), name=str(all_epochs[i]))
        for i in sampled
    ]
    slider_steps = [
        {
            "args": [[str(all_epochs[i])], {"frame": {"duration": 0}, "mode": "immediate",
                                            "transition": {"duration": 0}}],
            "label": str(all_epochs[i]),
            "method": "animate",
        }
        for i in sampled
    ]

    fig = go.Figure(
        data=_traces_at(sampled[0]),
        frames=frames,
        layout=go.Layout(
            title=f"{title_prefix}p={prime} | in-group PCA (fixed final-epoch basis)",
            scene=dict(
                xaxis=dict(title="Group PC1", range=_range(0)),
                yaxis=dict(title="Group PC2", range=_range(1)),
                zaxis=dict(title="Group PC3", range=_range(2)),
            ),
            legend=dict(x=1.01, y=1, font=dict(size=10)),
            template="plotly_white",
            height=700,
            updatemenus=[{
                "buttons": [
                    {"args": [None, {"frame": {"duration": 150}, "fromcurrent": True,
                                     "transition": {"duration": 0}}],
                     "label": "Play", "method": "animate"},
                    {"args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}],
                     "label": "Pause", "method": "animate"},
                ],
                "type": "buttons",
                "showactive": False,
                "x": 0.05, "y": 0, "xanchor": "left", "yanchor": "top",
            }],
            sliders=[{
                "steps": slider_steps,
                "currentvalue": {"prefix": "Epoch: ", "font": {"size": 13}},
                "pad": {"t": 50},
                "len": 0.85,
                "x": 0.1,
            }],
        ),
    )
    return fig

In [ ]:
# --- Run: p109 reference healthy ---
prime, model_seed, data_seed, label = MODELS[1]
v = family.get_variant(prime=prime, seed=model_seed, data_seed=data_seed)

fig = build_ingroup_animation(v, n_frames=60, title_prefix=f"{label} | ")
fig.show()